<a href="https://colab.research.google.com/github/vyperrrr99/TaxFineco/blob/main/Calcolo%20Tasse%20Fineco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Importa le librerie necessarie
import pandas as pd
from google.colab import drive
from google.colab import files
import io

print("--- Cella 1: Setup e Caricamento File ---")

# Monta Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive montato con successo!")
except Exception as e:
    print(f"Si è verificato un errore durante il montaggio di Google Drive: {e}")

# Inizializza le variabili che conterranno i dati
transactions_df = None
file_loaded = False

# --- Caricamento File Interattivo ---
print("\nClicca su 'Choose Files' e seleziona il file Excel con le movimentazioni.")
uploaded = files.upload()

if uploaded:
    file_name = next(iter(uploaded))
    print(f"\nCaricamento del file: '{file_name}'")
    try:
        # Legge i dati del file e li carica in un DataFrame
        transactions_df = pd.read_excel(io.BytesIO(uploaded[file_name]))
        file_loaded = True
        print("File caricato con successo nel DataFrame 'transactions_df'.")
    except Exception as e:
        print(f"ERRORE: Impossibile leggere il file. Assicurati che sia un file Excel valido. Dettagli: {e}")
else:
    print("\nOperazione annullata. Nessun file è stato selezionato.")


--- Cella 1: Setup e Caricamento File ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montato con successo!

Clicca su 'Choose Files' e seleziona il file Excel con le movimentazioni.


Saving Movimentazione azionaria.xlsx to Movimentazione azionaria (1).xlsx

Caricamento del file: 'Movimentazione azionaria (1).xlsx'
File caricato con successo nel DataFrame 'transactions_df'.


In [7]:
print("\n--- Cella 2: Validazione e Pulizia Dati ---")

# Controlla se il DataFrame dalla Cella 1 esiste ed è stato caricato correttamente
if 'file_loaded' in locals() and file_loaded:
    try:
        print("Inizio validazione e pulizia dati...")
        # --- VALIDAZIONE E PULIZIA DATI ---
        expected_columns = [
            'Data valuta', 'Descrizione', 'Titolo', 'Isin', 'Segno',
            'Quantita', 'Divisa', 'Prezzo', 'Cambio', 'Controvalore',
            'QTY', 'Val Unit €'
        ]
        missing_columns = [col for col in expected_columns if col not in transactions_df.columns]
        if missing_columns:
            raise ValueError(f"Colonne mancanti nel file: {', '.join(missing_columns)}")
        print("Tutte le colonne necessarie sono presenti.")

        transactions_df['Data valuta'] = pd.to_datetime(transactions_df['Data valuta'])
        numeric_cols = ['Quantita', 'Prezzo', 'Cambio', 'Controvalore', 'QTY', 'Val Unit €']
        for col in numeric_cols:
            transactions_df[col] = pd.to_numeric(transactions_df[col], errors='coerce')

        if transactions_df[numeric_cols].isnull().any().any():
            print("\nATTENZIONE: Trovati valori non numerici. Controlla le righe qui sotto. Queste righe verranno rimosse.")
            print(transactions_df[transactions_df[numeric_cols].isnull().any(axis=1)])
            transactions_df.dropna(subset=numeric_cols, inplace=True)

        transactions_df['Segno'] = transactions_df['Segno'].str.strip().str.upper()
        transactions_df['Titolo'] = transactions_df['Titolo'].str.strip()
        transactions_df['Isin'] = transactions_df['Isin'].str.strip()

        transactions_df = transactions_df.sort_values(by='Data valuta', ascending=True)
        transactions_df.reset_index(drop=True, inplace=True)

        print("\nElaborazione e pulizia dati completata con successo!")
        print("Anteprima dei dati processati:")
        display(transactions_df.head())

    except Exception as e:
        print(f"Si è verificato un errore durante l'elaborazione: {e}")
else:
    print("ERRORE: Esegui prima la Cella 1 per caricare il file.")



--- Cella 2: Validazione e Pulizia Dati ---
Inizio validazione e pulizia dati...
Tutte le colonne necessarie sono presenti.

Elaborazione e pulizia dati completata con successo!
Anteprima dei dati processati:


,Data valuta,Descrizione,Titolo,Isin,Segno,Quantita,Divisa,Prezzo,Cambio,Controvalore,QTY,Val Unit €
0,2020-10-28,Compravendita titoli,ONEOK,US6826801036,A,200,USD,29.41,1.18,"4,977.53",200,-24.89
1,2020-10-28,Compravendita titoli,ONEOK,US6826801036,A,100,USD,29.36,1.18,"2,484.56",100,-24.85
2,2020-11-10,Compravendita titoli,ONEOK,US6826801036,A,300,USD,28.32,1.19,"7,158.09",300,-23.86
3,2020-11-25,Compravendita titoli,ONEOK,US6826801036,A,200,USD,33.86,1.19,"5,708.45",200,-28.54
4,2021-03-18,Compravendita titoli,ONEOK,US6826801036,A,100,USD,49.60,1.19,"4,158.89",100,-41.59


In [9]:
print("\n--- Cella 3: Calcolo Plusvalenze/Minusvalenze e Salvataggio ---")

# Controlla se il DataFrame processato dalla Cella 2 esiste
if 'transactions_df' in locals() and not transactions_df.empty:
    try:
        print("Inizio calcolo plusvalenze/minusvalenze...")

        # Portfolio ora traccia anche il costo nella valuta originale
        portfolio = {}
        gain_loss_records = []

        for index, row in transactions_df.iterrows():
            isin = row['Isin']
            segno = row['Segno']
            descrizione = row['Descrizione'].strip() # Aggiunto per gestire l'aumento di capitale
            quantita = row['Quantita']
            controvalore_eur = row['Controvalore']
            prezzo_orig = row['Prezzo']
            divisa = row['Divisa']

            # Calcola il controvalore nella divisa originale per questa transazione
            controvalore_orig = quantita * prezzo_orig

            if descrizione == 'Aumento capitale':
                if isin in portfolio:
                    print(f"INFO: Rilevato Aumento di Capitale per {row['Titolo']} ({isin}) di {quantita} azioni.")
                    # Aumenta solo la quantità, il costo totale rimane invariato
                    portfolio[isin]['quantita'] += quantita
                else:
                    print(f"ATTENZIONE: Rilevato Aumento di Capitale per un titolo non in portafoglio: {row['Titolo']}. Operazione saltata.")
                continue # Passa alla riga successiva

            elif segno == 'A':
                if isin not in portfolio:
                    portfolio[isin] = {
                        'quantita': 0,
                        'costo_totale_eur': 0.0,
                        'costo_totale_orig': 0.0,
                        'divisa': divisa
                    }

                portfolio[isin]['quantita'] += quantita
                portfolio[isin]['costo_totale_eur'] += controvalore_eur
                portfolio[isin]['costo_totale_orig'] += controvalore_orig

            elif segno == 'V':
                if isin not in portfolio or portfolio[isin]['quantita'] < quantita:
                    print(f"ATTENZIONE: Vendita anomala di {row['Titolo']} ({isin}) il {row['Data valuta'].date()}. L'operazione verrà saltata.")
                    continue

                # Calcolo costo medio di carico unitario
                costo_medio_carico_eur = portfolio[isin]['costo_totale_eur'] / portfolio[isin]['quantita']
                costo_medio_carico_orig = portfolio[isin]['costo_totale_orig'] / portfolio[isin]['quantita']

                # Calcolo costo totale dell'operazione
                costo_operazione_eur = costo_medio_carico_eur * quantita
                costo_operazione_orig = costo_medio_carico_orig * quantita

                # Calcolo plus/minusvalenza
                plus_minus_valenza_eur = controvalore_eur - costo_operazione_eur
                plus_minus_valenza_orig = controvalore_orig - costo_operazione_orig

                # Calcolo prezzo di vendita unitario in EUR
                prezzo_vendita_eur = controvalore_eur / quantita if quantita != 0 else 0

                gain_loss_records.append({
                    'ISIN': isin, 'Titolo': row['Titolo'], 'Data Vendita': row['Data valuta'],
                    'Quantità Venduta': quantita,
                    'Divisa': divisa,
                    'Prezzo Medio Carico (€/azione)': costo_medio_carico_eur,
                    f'Prezzo Medio Carico ({divisa}/azione)': costo_medio_carico_orig,
                    'Prezzo Medio Vendita (€/azione)': prezzo_vendita_eur,
                    f'Prezzo Medio Vendita ({divisa}/azione)': prezzo_orig,
                    'Plus/Minusvalenza (€)': plus_minus_valenza_eur,
                    f'Plus/Minusvalenza ({divisa})': plus_minus_valenza_orig,
                    'Controvalore Vendita (€)': controvalore_eur,
                    'Costo Totale Carico (€)': costo_operazione_eur,
                    f'Controvalore Vendita ({divisa})': controvalore_orig,
                    f'Costo Totale Carico ({divisa})': costo_operazione_orig
                })

                # Aggiornamento del portafoglio
                portfolio[isin]['quantita'] -= quantita
                portfolio[isin]['costo_totale_eur'] -= costo_operazione_eur
                portfolio[isin]['costo_totale_orig'] -= costo_operazione_orig

                if portfolio[isin]['quantita'] < 1e-6: # Controllo per valori quasi a zero
                    portfolio[isin]['costo_totale_eur'] = 0.0
                    portfolio[isin]['costo_totale_orig'] = 0.0

        print("Calcolo completato.")

        if gain_loss_records:
            results_df = pd.DataFrame(gain_loss_records)
            total_gain_loss = results_df['Plus/Minusvalenza (€)'].sum()

            print("\n--- Riepilogo Operazioni di Vendita ---")
            pd.options.display.float_format = '{:,.2f}'.format
            display(results_df) # 'display' è preferibile a 'print' in Colab per i DataFrame

            print("\n-----------------------------------------")
            print(f"PLUSVALENZA/MINUSVALENZA TOTALE: {total_gain_loss:,.2f} €")
            print("-----------------------------------------")

            # --- SALVATAGGIO IN EXCEL ---
            output_filename = "Calcolo Tasse.xlsx"
            output_path = f"/content/drive/My Drive/{output_filename}"
            try:
                results_df.to_excel(output_path, index=False, engine='openpyxl')
                print(f"\nFile dei risultati salvato con successo in Google Drive:")
                print(f"Percorso: {output_path}")
            except Exception as e:
                print(f"\nERRORE durante il salvataggio del file Excel: {e}")

        else:
            print("\nNessuna operazione di vendita trovata nel file fornito.")

    except Exception as e:
        print(f"Si è verificato un errore durante il calcolo: {e}")
else:
    print("ERRORE: Esegui le celle 1 e 2 prima di calcolare le plusvalenze.")




--- Cella 3: Calcolo Plusvalenze/Minusvalenze e Salvataggio ---
Inizio calcolo plusvalenze/minusvalenze...
INFO: Rilevato Aumento di Capitale per SHOPIFY RG-A (CA82509L1076) di 315 azioni.
Calcolo completato.

--- Riepilogo Operazioni di Vendita ---


,ISIN,Titolo,Data Vendita,Quantità Venduta,Divisa,Prezzo Medio Carico (€/azione),Prezzo Medio Carico (USD/azione),Prezzo Medio Vendita (€/azione),Prezzo Medio Vendita (USD/azione),Plus/Minusvalenza (€),Plus/Minusvalenza (USD),Controvalore Vendita (€),Costo Totale Carico (€),Controvalore Vendita (USD),Costo Totale Carico (USD)
0,US6826801036,ONEOK,2021-09-30,501,USD,36.77,43.47,50.00,58.39,"6,629.96","7,476.30","25,050.00","18,420.04","29,253.39","21,777.09"
1,US6826801036,ONEOK,2021-09-30,399,USD,36.77,43.47,49.98,58.37,"5,271.63","5,944.23","19,941.48","14,669.85","23,287.66","17,343.43"
2,CA82509L1076,SHOPIFY RG-A,2022-12-01,100,USD,40.07,43.04,35.95,37.27,-411.88,-576.71,"3,595.41","4,007.29","3,727.00","4,303.71"
3,CA82509L1076,SHOPIFY RG-A,2022-12-01,100,USD,37.72,39.72,35.86,37.17,-186.18,-255.39,"3,585.82","3,772.00","3,717.06","3,972.45"
4,CA82509L1076,SHOPIFY RG-A,2022-12-01,1,USD,37.72,39.72,35.95,37.27,-1.77,-2.45,35.95,37.72,37.27,39.72
5,CA82509L1076,SHOPIFY RG-A,2022-12-01,114,USD,37.72,39.72,35.95,37.27,-201.31,-279.81,"4,098.77","4,300.08","4,248.78","4,528.59"
6,US6826801036,ONEOK,2024-06-13,430,USD,43.70,50.13,74.70,80.41,"13,329.22","13,020.96","32,119.16","18,789.94","34,576.29","21,555.32"
7,CA82509L1076,SHOPIFY RG-A,2024-08-19,100,USD,37.72,39.72,63.67,70.00,"2,595.11","3,027.55","6,367.11","3,772.00","7,000.00","3,972.45"
8,CA82509L1076,SHOPIFY RG-A,2024-10-01,285,USD,37.72,39.72,70.57,79.01,"9,360.96","11,194.97","20,111.16","10,750.20","22,516.46","11,321.49"
9,US6826801036,ONEOK,2024-12-31,600,USD,43.70,50.13,95.53,99.68,"31,098.26","29,732.86","57,316.78","26,218.52","59,810.06","30,077.20"



-----------------------------------------
PLUSVALENZA/MINUSVALENZA TOTALE: 67,484.00 €
-----------------------------------------

File dei risultati salvato con successo in Google Drive:
Percorso: /content/drive/My Drive/Calcolo Tasse.xlsx
